# Scraping: http://www.bbc.co.uk/news

Πάμε να πάρουμε περιεχόμενο από την αρχική σελίδα του BBC News. Θέλουμε:

* Τίτλους (Headlines)
* Συνόψεις (Summary)
* Υπερσυνδέσεις (link)

## Ξεκινώντας

Θέλουμε να **εισάγουμε τα απαραίτητα libraries**.

In [6]:
import requests
from bs4 import BeautifulSoup

Προχωράμε να **κατεβάσουμε την ιστοσελίδα** και *να την εισάγουμε στο BeautifulSoup*.

In [20]:
response = requests.get('https://www.bbc.com/news')
doc = BeautifulSoup(response.text, 'html.parser')

In [12]:
headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/70.0.3538.77 Safari/537.36"}

In [19]:
response = requests.get('http://www.bbc.co.uk/news', headers=headers)

In [21]:
doc

<!DOCTYPE html>
<html lang="en-GB"><head><meta charset="utf-8" data-next-head=""/><meta content="width=device-width" data-next-head="" name="viewport"/><title data-next-head="">BBC News - Breaking news, video and the latest top stories from the U.S. and around the world</title><meta content="News" data-next-head="" name="page.section"/><meta content="BBC News - Breaking news, video and the latest top stories from the U.S. and around the world" data-next-head="" property="og:title"/><meta content="BBC News - Breaking news, video and the latest top stories from the U.S. and around the world" data-next-head="" name="twitter:title"/><meta content="Visit BBC News for the latest news, breaking news, video, audio and analysis. BBC News provides trusted World, U.S. and U.K. news as well as local and regional perspectives. Also entertainment, climate, business, science, technology and health news." data-next-head="" name="description"/><meta content="Visit BBC News for the latest news, breaking

Συνήθως την σελίδα που αναλύουμε στο Beautiful Soup την ονομάζουμε `soup`, αλλά σήμερα θα κάνουμε τη διαφορά και θα την ονομάσουμε `doc`, για να θυμόμαστε ότι είναι ολόκληρο το *document*.

## Προσπάθεια 1η: Παίρνουμε απευθείας τα tags

Αν κοιτάξουμε την σελίδα, προσπαθούμε με το βελάκι να επιλέξουμε τα headlines, αλλά δεν **γίνεται**. Δεν μπορούμε να επιλέξουμε κανέναν τίτλο! Προφανώς πρόκειται για ολόκληρο το BLOCK του άρθρου ή κάτι τέτοιο?

In [11]:
 doc

<!DOCTYPE html>
<html class="no-js" lang="en-GB"><head><meta charset="utf-8"/><meta content="width=device-width, initial-scale=1" name="viewport"/><title data-rh="true">Home - BBC News</title><meta content="Visit BBC News for up-to-the-minute news, breaking news, video, audio and feature stories. BBC News provides trusted World and UK news as well as local and regional perspectives. Also entertainment, business, science, technology and health news." data-rh="true" name="description"/><meta content="#FFFFFF" data-rh="true" name="theme-color"/><meta content="light" data-rh="true" name="color-scheme"/><meta content="100004154058350" data-rh="true" property="fb:admins"/><meta content="3662195467332439" data-rh="true" property="fb:app_id"/><meta content="Visit BBC News for up-to-the-minute news, breaking news, video, audio and feature stories. BBC News provides trusted World and UK news as well as local and regional perspectives. Also entertainment, business, science, technology and health 

Ευτυχώς γνωρίζουμε λίγο HTML, οπότε μπορούμε να επιλέξουμε από το δεξί πλαίσιο που λέγεται Elements. Πάμε στο `h3` tag, το οποίο ξέρουμε ότι περιέχει τον τίτλο και το περιεχόμενο του.

Χμ, άρα μήπως να πάρουμε απλά όλα τα `h3` tags?

In [23]:
headlines = doc.find_all('h2')

for headline in headlines:
    print(headline.text)

Trump and King hold private talks after royals welcomed to White House with crowds and flypast
US political violence generates a familiar cycle - this time it's in overdrive
United Arab Emirates to quit oil cartel Opec
Why Sam Altman and his former hero Elon Musk are taking their toxic feud to court
'It's bizarre': Californians grapple with revelation that suspected Trump gunman was neighbour
Watch: Jimmy Kimmel defends 'expectant widow' joke after first lady criticism
Russian superyacht sails through Strait of Hormuz despite blockade
The other life of US soldier accused of betting on Maduro's removal
Suspected gunman aged 89 held in Greece after five wounded
Canada's Carney launches a sovereign wealth fund. What is it?
Ex-actor Nathan Chasing Horse jailed for at least 37 years for sexual assault
Man admits plotting attack on Taylor Swift concert in Vienna
Japan Airlines trials humanoid robots as ground handlers
More to explore
Watch as Sabastian Sawe's parents celebrate marathon recor

Αρκετά απλό, αλλά? ...λείπει το link και το summary.

Ok, οπότε μπορούμε να δοκιμάσουμε να πάρουμε όλα τα `a` tags, αλλά σίγουρα θα πάρουμε μαζί με αυτά που θέλουμε και αρκετά σκουπίδια που θα περιέχουν `a` tags - όπως π.χ. το περιεχόμενο στο footer της σελίδας κ.λπ. Πάμε να δούμε μήπως μέσα στα `a` tags του άρθρου υπάρχει κάποιο ξεχωριστό class?

Αν παρατηρήσουμε καλά θα δούμε ότι όντως είναι `class="gs-c-promo-heading nw-o-link-split__anchor gs-o-faux-block-link__overlay-link gel-pica-bold"`. Αυτό δεν είναι μόνο ένα class, είναι **ΠΟΛΛΑ classes**.

* `gs-c-promo-heading`
* `nw-o-link-split__anchor`
* `gs-o-faux-block-link__overlay-link`
* `gel-pica-bold`

Κάπου εδώ αρχίζουμε τις μαντεψιές. Ας δοκιμάσουμε πρώτα το `gs-c-promo-heading` που φαίνεται αρκετά λογικό να είναι το σωστό!

In [4]:
links = doc.find_all('a', { 'class': 'gs-c-promo-heading' })

for link in links:
    print(link.text)

Το αποτέλεσμα φαίνεται μια χαρά! Παίρνει το κείμενο του `h3`, γιατί το `h3` βρίσκεται μέσα στο `a` tag, αλλά δεν περιέχει το *link* του URL. Ας ρίξουμε μια ματιά στο `a` tag...

    <a class="gs-c-promo-heading nw-o-link-split__anchor gs-o-faux-block-link__overlay-link gel-pica-bold" href="/news/world-middle-east-39302560">
   
...το URL κρύβεται μέσα στην ιδιότητα `href`. Αν πάρουμε το link, είναι εύκολο να πάρουμε μετά το attribute, απλά χρησιμοποιούμε το`['href']`

In [5]:
links = doc.find_all('a', { 'class': 'gs-c-promo-heading' })

for link in links:
    print(link.text)
    print(link['href'])

Ωραίο, ε? Απλά τώρα μας μένει ένα τελευταίο πρόβλημα: **δεν έχουμε τα summaries**. Αλλά οκ, μπορούμε να χρησιμοποιήσουμε τον Inspector για να διαλέξουμε ένα...

    <p class="gs-c-promo-summary gel-long-primer gs-u-mt nw-c-promo-summary">Trade and Nato are high on the agenda as the much-anticipated Washington talks begin.</p>

Και πάλι, πρέπει να διαλέξουμε ένα από όλα αυτά, ας δοκιμάσουμε το `gs-c-promo-summary` που φαίνεται καλό.

In [ ]:
summaries = doc.find_all('p', { 'class': 'gs-c-promo-summary' })

for summary in summaries:
    print(summary.text)

The prime minister spoke after two nights of clashes between Palestinian protesters and police.
The prime minister spoke after two nights of clashes between Palestinian protesters and police.
Most of the victims of Saturday's blasts which killed more than 60 near a Kabul school were girls.
She said "women and especially women of colour have seen a generation of economic gain wiped out".
The 49-year-old Tesla CEO appeared as a guest host on the US comedy series Saturday Night Live.
Kentucky Derby winner Medina Spirit, who provided trainer Bob Baffert with a record seventh win in the race, has failed a drugs test.
Doctors in India are reporting a rash of a rare infection in Covid patients which is making them blind.
Most of the Chinese Long March-5b rocket reportedly disintegrated as it re-entered the atmosphere.
Manchester United's efforts to secure a lucrative new training kit deal have suffered a setback after a local firm withdrew over concerns it could be boycotted by fans protestin

Ωραία, αλλά **τώρα έχουμε κολλήσει:** δεν υπάρχει τρόπος να συνδυάσουμε και τα headlines και τα links με τα summaries και ακόμη και αν υπήρχε (ίσως με `zip`), δεν θα ήμασταν σίγουροι ότι είναι σωστό το ματσάρισμα.

Άρα τι κάνουμε τώρα?

## Προσπάθεια 2η: Parent elements

Όταν επιλέγουμε ένα element - ένα link και το κείμενο του, ή μια λίστα με headlines - μας ενδιαφέρει μόνο το element που κοιτάμε. Μερικές φορές όμως, **πρέπει να πάρουμε πολλά διαφορετικά elements την ίδια στιγμή.** Όταν συμβαίνει αυτό πρέπει να δούμε τι κοινό έχουν όλα αυτά μαζί.

Αν κοιτάξουμε το summary, το link και τον τίτλο, ίσως βρούμε κάτι σαν το παρακάτω. **Είναι γιγάντιο, αλλά τι να κάνουμε αυτό θέλουμε.**

	<div class="gs-c-promo nw-c-promo gs-o-faux-block-link gs-u-pb gs-u-pb+@m gs-c-promo--inline gs-c-promo--stacked@m nw-u-w-auto gs-c-promo--flex" data-entityid="container-top-stories#3">
		<div class="gs-c-promo-image gs-u-display-none gs-u-display-inline-block@xs gel-1/2@xs gel-1/1@m">
			<div class="gs-o-media-island">
				<div class="gs-o-responsive-image gs-o-responsive-image--16by9"><img alt="Yemeni police gather round bodies of Somali refugees (17/03/19)" class="qa-lazyload-image lazyautosizes lazyloaded" data-sizes="auto" data-srcset="https://ichef.bbci.co.uk/live-experience/cps/240/cpsprodpb/14F5A/production/_95205858_hi038526789.jpg 240w, https://ichef.bbci.co.uk/live-experience/cps/320/cpsprodpb/14F5A/production/_95205858_hi038526789.jpg 320w, https://ichef.bbci.co.uk/live-experience/cps/480/cpsprodpb/14F5A/production/_95205858_hi038526789.jpg 480w, https://ichef.bbci.co.uk/live-experience/cps/624/cpsprodpb/14F5A/production/_95205858_hi038526789.jpg 624w, https://ichef.bbci.co.uk/live-experience/cps/800/cpsprodpb/14F5A/production/_95205858_hi038526789.jpg 800w" data-widths="[240,320,480,624,800]" sizes="177px" src="data:image/gif;base64,R0lGODlhAQABAIAAAAAAAP///yH5BAEAAAAALAAAAAABAAEAAAIBRAA7" srcset="https://ichef.bbci.co.uk/live-experience/cps/240/cpsprodpb/14F5A/production/_95205858_hi038526789.jpg 240w, https://ichef.bbci.co.uk/live-experience/cps/320/cpsprodpb/14F5A/production/_95205858_hi038526789.jpg 320w, https://ichef.bbci.co.uk/live-experience/cps/480/cpsprodpb/14F5A/production/_95205858_hi038526789.jpg 480w, https://ichef.bbci.co.uk/live-experience/cps/624/cpsprodpb/14F5A/production/_95205858_hi038526789.jpg 624w, https://ichef.bbci.co.uk/live-experience/cps/800/cpsprodpb/14F5A/production/_95205858_hi038526789.jpg 800w"></div>
			</div>
		</div>
		<div class="gs-c-promo-body gel-1/2@xs gel-1/1@m gs-u-mt@m">
			<div>
				<a class="gs-c-promo-heading nw-o-link-split__anchor gs-o-faux-block-link__overlay-link gel-pica-bold" href="/news/world-middle-east-39302560">
				<h3 class="gs-c-promo-heading__title gel-pica-bold nw-o-link-split__text">Attack on Yemen migrant boat kills 42</h3></a>
				<p class="gs-c-promo-summary gel-long-primer gs-u-mt nw-c-promo-summary">It is unclear who was behind a helicopter attack which killed 42 refugees and injured 80.</p>
			</div>
			<ul class="gs-o-list-inline gs-o-list-inline--divided gel-brevier gs-u-mt-">
				<li><span class="gs-c-timestamp gs-o-bullet gs-o-bullet- nw-c-timestamp"><span class="gs-o-bullet__icon gel-icon"><svg viewbox="0 0 32 32">
				<polygon points="17,15.4 17,6 15,6 15,16.6 23.8,21.7 24.8,19.9"></polygon>
				<path d="M16,4c6.6,0,12,5.4,12,12c0,6.6-5.4,12-12,12S4,22.6,4,16C4,9.4,9.4,4,16,4 M16,0C7.2,0,0,7.2,0,16c0,8.8,7.2,16,16,16 s16-7.2,16-16C32,7.2,24.8,0,16,0L16,0z"></path></svg></span><time class="gs-o-bullet__text date qa-status-date relative-time" data-datetime="1h" data-seconds="1489768430" data-timestamp-inserted="true" datetime="2017-03-17T16:33:50.000Z">48 minutes ago</time></span></li>
				<li>
					<a aria-label="From Middle East" class="gs-c-section-link gs-c-section-link--truncate nw-c-section-link nw-o-link nw-o-link--no-visited-state" href="/news/world/middle_east"><span aria-hidden="true">Middle East</span></a>
				</li>
			</ul>
		</div>
	</div>

To πάνω πάνω κομμάτι λέγεται **parent element**, όλα τα άλλα elements βρίσκονται μέσα σε αυτό. Αν θέλουμε να το πάρουμε όλο μαζί, τότε πρέπει πρώτα να παίρνουμε τον κάθε γονέα (parent) κάθε *είδηση* και μετά να πάρουμε τα επιμέρους κομμάτια που υπάρχουν μέσα π.χ. headline, links, εικόνες κ.λπ.

Tο κομμάτι του class είναι `class="gs-c-promo nw-c-promo gs-o-faux-block-link gs-u-pb gs-u-pb+@m gs-c-promo--inline gs-c-promo--stacked@m nw-u-w-auto gs-c-promo--flex" data-entityid="container-top-stories#3"`, το οποίο εντάξει όντως είναι κάπως τρομακτικό, αλλά θα δοκιμάσουμε το `gs-c-promo` και ίσως δουλέψει.

In [24]:
stories = doc.find_all('div', { 'class': 'sc-225578b-0 ezQaGx' })
for story in stories:
    print(story.text)

LIVETrump and King hold private talks after royals welcomed to White House with crowds and flypastLater, the monarch is due to make an historic address to US lawmakers.
US political violence generates a familiar cycle - this time it's in overdriveIn modern America, it seems violence of this kind has become an ever-present storm that can strike anywhere and at any moment.16 hrs agoUS & Canada
United Arab Emirates to quit oil cartel OpecThe UAE leaving Opec is seen as a major blow and potential death knell for the oil cartel.39 mins agoBusiness
Why Sam Altman and his former hero Elon Musk are taking their toxic feud to courtThe battle between the AI big hitters has largely played out on social media. Now it is coming to the courtroom.26 mins ago
'It's bizarre': Californians grapple with revelation that suspected Trump gunman was neighbourPeople living near the man charged with the attempted assassination of Trump  say they were shocked to see their neighbour on TV. 5 hrs agoUS & Canada
W



Προφανώς δεν μπορούμε να χρησιμοποιήσουμε το `.text` γιατί θα πάρει *τα πάντα* που περιέχουν κείμενο, και το headline *και* το summary. Αντιθέτως αυτό που πρέπει να κάνουμε είναι

* ΒΗΜΑ ΕΝΑ: Χρησιμοποιούμε το doc για να πάρουμε την είδηση
* ΒΗΜΑ ΔΥΟ: Χρησιμοποιούμε την είδηση για να πάρουμε τον τίτλο
* ΒΗΜΑ ΤΡΙΑ: Χρησιμοποιούμε την είδηση για να πάρουμε τον link
* ΒΗΜΑ ΤΕΣΣΕΡΑ: Χρησιμοποιούμε την είδηση για να πάρουμε την σύνοψη

### ΒΗΜΑ ΕΝΑ: Χρησιμοποιούμε το doc για να πάρουμε την είδηση (story)

In [25]:
stories = doc.find_all('div', { 'class': 'sc-225578b-0 ezQaGx' })
for story in stories:
    print("This is a story")

This is a story
This is a story
This is a story
This is a story
This is a story
This is a story
This is a story
This is a story
This is a story
This is a story
This is a story
This is a story
This is a story
This is a story
This is a story
This is a story
This is a story
This is a story
This is a story
This is a story
This is a story
This is a story
This is a story
This is a story
This is a story
This is a story
This is a story
This is a story
This is a story
This is a story
This is a story
This is a story
This is a story
This is a story
This is a story
This is a story
This is a story
This is a story
This is a story
This is a story
This is a story
This is a story
This is a story
This is a story
This is a story
This is a story
This is a story
This is a story
This is a story


## ΒΗΜΑ ΔΥΟ + ΤΡΙΑ: Χρησιμοποιούμε την είδηση για να πάρουμε τον τίτλο

Τώρα μπορούμε να κάνουμε ακριβώς το ίδιο για να πάρουμε τον τίτλο και το link, και μετά με το `['href']` να πάρουμε το link του URL.

In [26]:
stories = doc.find_all('div', { 'class': 'sc-225578b-0 ezQaGx' })
for story in stories:
    print("THIS IS A STORY")
    headline = story.find('h2')
    print(headline.text)
    link = story.find('a')
    print(link['href'])


 #   headlines = doc.find_all('h3')

#for headline in headlines:
   # print(headline.text)

THIS IS A STORY
Trump and King hold private talks after royals welcomed to White House with crowds and flypast
https://www.bbc.com/news/live/c4g5lly7qg8t
THIS IS A STORY
US political violence generates a familiar cycle - this time it's in overdrive
/news/articles/cje4eeyq27lo
THIS IS A STORY
United Arab Emirates to quit oil cartel Opec
/news/articles/cj4pxwlr52yo
THIS IS A STORY
Why Sam Altman and his former hero Elon Musk are taking their toxic feud to court
/news/articles/cn8dedv8w8xo
THIS IS A STORY
'It's bizarre': Californians grapple with revelation that suspected Trump gunman was neighbour
/news/articles/c392knk3w2jo
THIS IS A STORY
Watch: Jimmy Kimmel defends 'expectant widow' joke after first lady criticism
/news/videos/c62d4n5rzr7o
THIS IS A STORY
Russian superyacht sails through Strait of Hormuz despite blockade
/news/articles/cm2pn8zdxdjo
THIS IS A STORY
The other life of US soldier accused of betting on Maduro's removal
/news/articles/clyxd5wrr0wo
THIS IS A STORY
Suspected 

## ΒΗΜΑ ΤΕΣΣΕΡΑ: Χρησιμοποιούμε την είδηση για να πάρουμε την σύνοψη

Το ίδιο και πάλι! Αυτή τη φορά ψάχνουμε τις παραγράφους `p`.

In [31]:
stories = doc.find_all('div', { 'class': 'sc-225578b-0 ezQaGx' })
for story in stories:
    print("THIS IS A STORY")
    headline = story.find('h2')
    print(headline.text)
    link = story.find('a')
    print(link['href'])
    summary = story.find('p')
    if summary:
        print(summary.text)


THIS IS A STORY
Trump and King hold private talks after royals welcomed to White House with crowds and flypast
https://www.bbc.com/news/live/c4g5lly7qg8t
Later, the monarch is due to make an historic address to US lawmakers.
THIS IS A STORY
US political violence generates a familiar cycle - this time it's in overdrive
/news/articles/cje4eeyq27lo
In modern America, it seems violence of this kind has become an ever-present storm that can strike anywhere and at any moment.
THIS IS A STORY
United Arab Emirates to quit oil cartel Opec
/news/articles/cj4pxwlr52yo
The UAE leaving Opec is seen as a major blow and potential death knell for the oil cartel.
THIS IS A STORY
Why Sam Altman and his former hero Elon Musk are taking their toxic feud to court
/news/articles/cn8dedv8w8xo
The battle between the AI big hitters has largely played out on social media. Now it is coming to the courtroom.
THIS IS A STORY
'It's bizarre': Californians grapple with revelation that suspected Trump gunman was neigh

### Λείπουν elements

Αυτό μας έλειπε τώρα, ένα error! Συγκεκριμένα λέει ότι ...

    ---------------------------------------------------------------------------
    AttributeError                            Traceback (most recent call last)
    <ipython-input-20-e55795264040> in <module>()
          7     print(link['href'])
          8     summary = story.find('p')
    ----> 9     print(summary.text)

    AttributeError: 'NoneType' object has no attribute 'text'

Εφόσον το λάθος εμφανίστηκε από την στιγμή που βάλαμε το κομμάτι με το `summary` part, υποθέτουμε ότι το πρόβλημα είναι ότι δεν έχει κάθε μια **ιστορία από ένα summary**. Πώς θα το λύσουμε αυτό???!!!

Λοιπόν, μπορούμε να *ρωτάμε αν (if) η ιστορία έχει summary*. Αν έχει τότε μπορούμε να το χρησιμοποιήσουμε. Αν δεν έχει το αγνοούμε. **Είναι ένα απλό `if` statement**.

In [ ]:
stories = doc.find_all('div', { 'class': 'gs-c-promo' })
for story in stories:
    print("THIS IS A STORY")
    headline = story.find('h3')
    print(headline.text)
    link = story.find('a')
    print(link['href'])
    summary = story.find('p')
    if summary:
        print(summary.text)

THIS IS A STORY
Netanyahu defends handling of Jerusalem clashes
/news/world-middle-east-57049126
The prime minister spoke after two nights of clashes between Palestinian protesters and police.
THIS IS A STORY
Afghan families bury schoolgirls killed by blasts
/news/world-asia-57046527
Most of the victims of Saturday's blasts which killed more than 60 near a Kabul school were girls.
THIS IS A STORY
Meghan warns of pandemic impact on women 
/news/world-57047169
She said "women and especially women of colour have seen a generation of economic gain wiped out".
THIS IS A STORY
Elon Musk reveals he has Asperger's
/news/world-us-canada-57045770
The 49-year-old Tesla CEO appeared as a guest host on the US comedy series Saturday Night Live.
THIS IS A STORY
Kentucky Derby winner fails drugs test
/sport/horse-racing/57049450
Kentucky Derby winner Medina Spirit, who provided trainer Bob Baffert with a record seventh win in the race, has failed a drugs test.
THIS IS A STORY
A bizarre tale of cannabi

## Ας το σώσουμε ως CSV

Τώρα που έχουμε όλα τα στοιχεία που θέλαμε μπορούμε να τα μετατρέψουμε σε ένα CSV. Υπάρχουν 3 βήματα για τη μετατροπή σε CSV:
    
1. **Ξεκινάμε με μια άδεια λίστα:** Κάθε ιστορία που βρίσκουμε, την προσθέτουμε στη λίστα
2. **Φτιάχνουμε ένα λεξικό** για κάθε μια ιστορία
3. **Μετατρέπουμε τη λίστα σε ένα DataFrame**, μετά
4. **Εξάγουμε το DataFrame σε ένα CSV**

Η δημιουργία του λεξικού μπορεί να είναι λίγο περίπλοκη, οπότε ας δούμε **2 διαφορετικούς τρόπους να το κάνουμε**.

### Μέθοδος 1η: Όλα μαζί

Θα φτιάξουμε το λεξικό `story_dict` απευθείας και μετά θα το προσθέσουμε στη λίστα `stories_list`.

In [32]:
# Ξεκινάμε με μια άδεια λίστα
stories_list = []
stories = doc.find_all('div', { 'class': 'sc-225578b-0 ezQaGx' })
for story in stories:
    headline = story.find('h2')
    link = story.find('a')
    summary = story.find('p')
    # Έχει η ιστορία summary?
    if summary:
        summary_text = summary.text
    else:
        summary_text = ''

        # Φτιάξε ένα λεξικό αν ΕΧΕΙ summary
    story_dict = {
            'headline': headline.text,
            'url': link['href'],
            'summary': summary_text
        }

    # Πρόσθεσε το λέξικο στη λίστα
    stories_list.append(story_dict)

print(stories_list)

# Τώρα που τελειώσαμε μετέτρεψε το σε CSV και αποθήκευσε το.
# Αν δεν χρησιμοποιήσετε το index=False, θα έχετε ένα άσχημο dataframe!
import pandas as pd
df = pd.DataFrame(stories_list)
#df.to_csv("bbc.csv", index=False)

[{'headline': 'Trump and King hold private talks after royals welcomed to White House with crowds and flypast', 'url': 'https://www.bbc.com/news/live/c4g5lly7qg8t', 'summary': 'Later, the monarch is due to make an historic address to US lawmakers.'}, {'headline': "US political violence generates a familiar cycle - this time it's in overdrive", 'url': '/news/articles/cje4eeyq27lo', 'summary': 'In modern America, it seems violence of this kind has become an ever-present storm that can strike anywhere and at any moment.'}, {'headline': 'United Arab Emirates to quit oil cartel Opec', 'url': '/news/articles/cj4pxwlr52yo', 'summary': 'The UAE leaving Opec is seen as a major blow and potential death knell for the oil cartel.'}, {'headline': 'Why Sam Altman and his former hero Elon Musk are taking their toxic feud to court', 'url': '/news/articles/cn8dedv8w8xo', 'summary': 'The battle between the AI big hitters has largely played out on social media. Now it is coming to the courtroom.'}, {'hea

In [33]:
df.head()

,headline,url,summary
0,Trump and King hold private talks after royals...,https://www.bbc.com/news/live/c4g5lly7qg8t,"Later, the monarch is due to make an historic ..."
1,US political violence generates a familiar cyc...,/news/articles/cje4eeyq27lo,"In modern America, it seems violence of this k..."
2,United Arab Emirates to quit oil cartel Opec,/news/articles/cj4pxwlr52yo,The UAE leaving Opec is seen as a major blow a...
3,Why Sam Altman and his former hero Elon Musk a...,/news/articles/cn8dedv8w8xo,The battle between the AI big hitters has larg...
4,'It's bizarre': Californians grapple with reve...,/news/articles/c392knk3w2jo,People living near the man charged with the at...


### Μέθοδος 2η: Γεμίζοντας τα κενά

Για αυτή τη μέθοδο θα φτιάξουμε πρώτα το λεξικό μας `story_dict` και μετά θα το γεμίζουμε με τα στοιχεία που υπάρχουν.

In [ ]:
# Ξεκινάμε με μια άδεια λίστα
stories_list = []
stories = doc.find_all('div', { 'class': 'gs-c-promo' })
for story in stories:
    # Φτιάχνουμε ένα λεξικό χωρίς τίποτα
    story_dict = {}
    headline = story.find('h2')
    if headline:
        story_dict['headline'] = headline.text
    link = story.find('a')
    if link:
        story_dict['url'] = link['href']
    summary = story.find('p')
    if summary:
        story_dict['summary'] = summary.text
    # Προσθέτουμε το λεξικό στη λίστα
    stories_list.append(story_dict)

# Τώρα που τελειώσαμε μετέτρεψε το σε CSV και αποθήκευσε το.
# Αν δεν χρησιμοποιήσετε το index=False, θα έχετε ένα άσχημο dataframe!
import pandas as pd
df = pd.DataFrame(stories_list)
df.to_csv("bbc.csv", index=False)

In [45]:
stories_list = []

for i in range(1,6):
  url = "https://www.capital.gr/oikonomia?page=" + str(i)
  response = requests.get(url)
  doc = BeautifulSoup(response.text, 'html.parser')

  stories = doc.find_all('div', { 'class': 'article' })
  for story in stories:
    # Φτιάχνουμε ένα λεξικό χωρίς τίποτα
    story_dict = {}
    headline = story.find('h2')
    if headline:
        story_dict['headline'] = headline.text
    link = story.find('a')
    if link:
        story_dict['url'] = link['href']
    summary = story.find('p')
    if summary:
        story_dict['summary'] = summary.text
    # Προσθέτουμε το λεξικό στη λίστα
    stories_list.append(story_dict)

In [42]:
#stories_list

In [46]:
df = pd.DataFrame(stories_list)

In [47]:
df.tail()

,headline,url,summary
84,\nΕπίτροπος Ενέργειας: Μακροχρόνια και απρόβλε...,/oikonomia/3986851/epitropos-energeias-makroxr...,\r\nΤις πρώτες 50 ημέρες της σύγκρουσης στη Μέ...
85,\nΚ. Πιερρακάκης: Είναι παράνομο αυτό που λέει...,/oikonomia/3986840/k-pierrakakis-einai-paranom...,\r\nΑπάντηση στις προτάσεις για ευρύτερη αξιοπ...
86,\nΝ. Κεραμέως: Το πλεόνασμα επιστρέφει στην κο...,/oikonomia/3986811/n-kerameos-to-pleonasma-epi...,"\r\nΌπως αναφέρει, τα μέτρα περιλαμβάνουν νέα ..."
87,"\nΓερμανία: Ανάπτυξη μόλις 0,5% για το 2026 βλ...",/oikonomia/3986805/germania-anaptuxi-molis-0-5...,\r\nΗ ομοσπονδιακή κυβέρνηση αναθεώρησε σήμερα...
88,\nΓ. Μπρατάκος: Θετικά τα μέτρα για τις επιχει...,/oikonomia/3986797/g-mpratakos-thetika-ta-metr...,"\r\nΘετικά υποδέχεται, το Εμπορικό και Βιομηχα..."
